In [0]:
# ---- Config ----
CATALOG = "workspace"
BRONZE_SCHEMA = "careerpulse_bronze"
SILVER_SCHEMA = "careerpulse_silver"
BRONZE_TABLE  = f"{CATALOG}.{BRONZE_SCHEMA}.job_postings_raw"
SILVER_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.job_postings"

min_titles_missing = 0.01
min_descriptions_missing = 0.01
min_future_posted = 0

In [0]:
from pyspark.sql.functions import (
    col, 
    from_json, 
    explode, 
    sha2, 
    concat_ws, 
    lower, 
    substring, 
    to_timestamp, 
    lit,
    current_timestamp
    )
from pyspark.sql.types import *

muse_schema = StructType([
    StructField("page", LongType()),
    StructField("page_count", LongType()),
    StructField("items_per_page", LongType()),
    StructField("took", LongType()),
    StructField("timed_out", BooleanType()),
    StructField("total", LongType()),
    StructField("results", ArrayType(StructType([
        StructField("contents", StringType()),
        StructField("name", StringType()),
        StructField("type", StringType()),
        StructField("publication_date", StringType()),  # parse to timestamp later
        StructField("short_name", StringType()),
        StructField("model_type", StringType()),
        StructField("id", LongType()),
        StructField("locations", ArrayType(StructType([
            StructField("name", StringType())
        ]))),
        StructField("categories", ArrayType(StructType([
            StructField("name", StringType()),
            StructField("short_name", StringType())
        ]))),
        StructField("levels", ArrayType(StructType([
            StructField("name", StringType()),
            StructField("short_name", StringType())
        ]))),
        StructField("tags", ArrayType(StringType())),
        StructField("refs", StructType([
            StructField("landing_page", StringType())
        ])),
        StructField("company", StructType([
            StructField("id", LongType()),
            StructField("short_name", StringType()),
            StructField("name", StringType())
        ]))
    ])))
])

bronze = spark.table(BRONZE_TABLE)

parsed = (
    bronze
    .withColumn("parsed", from_json(col("payload_json"), muse_schema))
    .withColumn("job", explode(col("parsed.results")))
)


In [0]:
from pyspark.sql import functions as F

# Create the silver table from the parsed bronze
silver = (
    parsed
    .select(
        col("job.id").cast("string").alias("source_job_id"),          # numeric job id from The Muse
        col("job.company.name").alias("company_name"),                # company name (nested struct)
        col("job.name").alias("title"),                               # job title is "name" in Muse
        col("job.contents").alias("description"),                     # HTML job description
        lit(None).cast("boolean").alias("remote"),                    # Muse payload here doesn't include a remote flag reliably
        col("job.tags").alias("tags"),                                # list of tags
        F.get(col("job.locations"), 0).getField("name").alias("location"),            # take first location name
        F.when(
            F.lower(F.get(col("job.categories"), 0).getField("name")).isin(["unknown", "none", ""]), None
            ).otherwise( #replace "Unknown" with null"
                F.get(col("job.categories"), 0).getField("name")
                ).alias("category"),
        F.get(col("job.categories"), 0).getField("short_name").alias("category_short"),
        F.get(col("job.levels"), 0).getField("name").alias("level"),
        F.get(col("job.levels"), 0).getField("short_name").alias("level_short"),
        to_timestamp(col("job.publication_date")).alias("posted_at"), # ISO-8601 string -> timestamp
        col("ingestion_date")                                         # from Bronze
    )
).withColumn(
    "posting_id",
    sha2(
        concat_ws(
            "||",
            lower(col("title")),
            lower(col("company_name")),
            lower(col("location")),
            substring(lower(col("description")), 1, 200)              # hash based on description snippet (more stable than ingestion_date)
        ),
        256
    )
)

# Data quality checks on the incoming silver data
total = silver.count()
missing_title = silver.filter(col("title").isNull() | (col("title") == "")).count()
missing_desc  = silver.filter(col("description").isNull() | (col("description") == "")).count()
future_posted = silver.filter(col("posted_at") > current_timestamp()).count()

if total == 0:
    raise ValueError("Silver incoming is empty")

if missing_title / total > min_titles_missing:
    raise ValueError(f"Too many missing titles: {missing_title}/{total}")

if missing_desc / total > min_descriptions_missing:
    raise ValueError(f"Too many missing descriptions: {missing_desc}/{total}")

if future_posted > min_future_posted:
    raise ValueError(f"Found {future_posted} rows with posted_at in the future")


silver.dropDuplicates(["posting_id"]).createOrReplaceTempView("silver_incoming")

# if the table doesn't exist, create it using the incoming schema from the incoming table
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
          USING DELTA
          AS
          SELECT * FROM silver_incoming
          WHERE 1 = 0
          """)

# merges non-dup rows into the silver table
spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS t
        USING silver_incoming AS s
        ON t.posting_id = s.posting_id
        -- WHEN MATCHED THEN UPDATE SET * --uncomment for upsert
        WHEN NOT MATCHED THEN INSERT *
        """)


In [0]:
spark.table(SILVER_TABLE).filter(col("category")=="Unknown").display()

In [0]:
%sql
--show how many total, unique and dup postings are in the table
SELECT
  COUNT(*) AS n_rows, --total
  COUNT(DISTINCT posting_id) AS distinct_posting_ids, --unique
  (COUNT(*) - COUNT(DISTINCT posting_id)) AS duplicate_posting_ids --duplicates
FROM workspace.careerpulse_silver.job_postings;



In [0]:
%sql
--show a sample of the silver table
SELECT *
FROM workspace.careerpulse_silver.job_postings
ORDER BY posted_at DESC
LIMIT 5;


In [0]:
silver.filter(silver.remote.isNull()).count()

In [0]:
%sql
SELECT 
  category, 
  COUNT(*) as n
FROM workspace.careerpulse_silver.job_postings
GROUP BY category
ORDER BY n DESC